# MMM Multidimensional Example Notebook

In this notebook, we replicate the multidimensional MMM example from the PyMC Marketing docs. We fit a hierarchical model across two geographies with separate channels, inspect prior/posterior predictives, recover parameters, make out-of-sample forecasts, and demonstrate budget optimization.

In [ ]:
import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import seaborn as sns
import xarray as xr
from pymc_extras.prior import Prior

from pymc_marketing.mmm.multidimensional import MMM
from pymc_marketing.paths import data_dir
from pymc_marketing.special_priors import  LogNormalPrior

warnings.filterwarnings("ignore", category=UserWarning)

az.style.use("arviz-darkgrid")
plt.rcParams["figure.figsize"] = [12, 7]
plt.rcParams["figure.dpi"] = 100
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 8

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"



# environment setup
import sys
sys.path.insert(0, ".")
import yaml
import logging
from typing import Dict, Any


# pymc-labs imports
# pymc_marketing upgrade imports
from pymc_extras.prior import Prior
from pymc_extras.prior import VariableFactory
from pymc_marketing.mmm.components.adstock import GeometricAdstock
from pymc_marketing.mmm.components.saturation import LogisticSaturation
from pymc_marketing.mmm.components.adstock import NoAdstock
from pymc_marketing.mmm.components.saturation import NoSaturation

# helper libraries
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytensor 
import jax 
import pytensor.tensor as pt
import numpyro 
import pandas as pd
from pymc_marketing.mmm.components.saturation import NoSaturation


# Import custom functions

from b2b_src.ETL import load_and_standardize_data
from b2b_src.Prior_upgrade import PerChannelDistributionFactory
from b2b_src.contributions import compare_predictions

# Import multidimensional contribution calculator
from multi_src import get_individual_contributions_multidim

# Import pre-decorated functions (Pattern 1 - Recommended!)
from multi_src import (
    # Calculation functions
    calculate_roas,
    calculate_roas_time_series,
    calculate_marginal_roas,
    # Plotting functions
    plot_roas_analysis,
    plot_roas_time_series,
    plot_all_contributions,
)

# Import helper functions
from multi_src import slice_to_geo

# Keep original imports for functions not yet decorated
from b2b_src.visual_tools import plot_sales_and_channels
from b2b_src.visual_tools import plot_prior_predictive_with_log_reversion
from b2b_src.visual_tools import plot_posterior_predictive_with_log_reversion
from b2b_src.visual_tools import plot_response_curves
from b2b_src.contributions import analyze_revenue_vs_spend_levels
from b2b_src.utils import plot_trace_with_channel_names      
from b2b_src.utils import summary_with_channel_names         
from pymc_marketing.logging_config import setup_logging


# Import utils functions
from b2b_src.utils import create_adstock_from_config
from b2b_src.utils import create_saturation_from_config

seed: int = sum(map(ord, "mmm_multidimensional"))
rng: np.random.Generator = np.random.default_rng(seed=seed)

# Configuration Setup

**IMPORTANT:** This cell must be run first to configure column names for the entire notebook.

All functions in this notebook will automatically use these column names instead of hardcoded values.

In [ ]:
# =============================================================================
# LOAD CONFIGURATION (DO THIS FIRST!)
# =============================================================================
from b2b_src.config_loader import load_geo_config, print_config_summary
from b2b_src.column_config import ColumnConfig, set_default_column_config

# Load config from YAML file
config = load_geo_config("geo_config.yml")

# Create column configuration
cols = ColumnConfig(config)

# Set as global default - all functions will use this
set_default_column_config(cols)

# Print summary to verify
print("✓ Configuration loaded successfully!")
print(f"  Target column: {cols.target}")
print(f"  Dimension column: {cols.dimension}")
print(f"  Date column: {cols.date}")
print(f"  Channels: {cols.channels}")
print(f"  Controls: {cols.controls}")
print("\n" + "="*70)
print("All functions will now use these column names automatically!")
print("="*70)

In [ ]:
# Read Data
data_path = data_dir / "test_geo_model_dataset.csv"

data_df = pd.read_csv(data_path, parse_dates=["date"]) 
print(data_df.info())
data_df.head()

For our setup, imagine we are selling one product in two geographies (geo_a and geo_b) with two channels. Channel 1 is usually-on; channel 2 is tactical. We'll pool parameters across geos where reasonable and leave others unpooled to illustrate hierarchical modeling.

In [ ]:
# =============================================================================
# CHANNEL SPENDS BY GEO PLOT
# =============================================================================

from b2b_src.visual_tools import plot_channel_spends_by_geo

# OPTION 1: Basic usage with automatic column detection from config
# ------------------------------------------------------------------
fig = plot_channel_spends_by_geo(data_df)
plt.show()

# # OPTION 2: With custom title and dimension label override
# # ----------------------------------------------------------
# fig = plot_channel_spends_by_geo(
#     data_df,
#     custom_title="Marketing Investment Analysis",
#     dimension_label_override="BRAND"  # All geos will show as "BRAND"
# )
# plt.show()

# # OPTION 3: With automatic dimension labels (shows actual geo names)
# # -------------------------------------------------------------------
# fig = plot_channel_spends_by_geo(
#     data_df,
#     custom_title="Channel Performance by Region"
#     # No dimension_label_override, so shows "BRAND_A", "BRAND_B"
# )
# plt.show()

# # OPTION 4: With custom colors and figure size
# # ---------------------------------------------
# fig = plot_channel_spends_by_geo(
#     data_df,
#     figsize=(14, 10),
#     bar_width=10,
#     bar_colors=["#FF6B6B", "#4ECDC4", "#45B7D1"],
#     custom_title="Weekly Channel Spends",
#     dimension_label_override="MARKET"
# )
# plt.show()

# # OPTION 5: Explicit column specification (if not using config)
# # --------------------------------------------------------------
# fig = plot_channel_spends_by_geo(
#     data_df,
#     channel_cols=['Spend_channel_1', 'Spend_channel_2'],
#     target_col='NetGMV',
#     dimension_col='Groups',
#     date_col='date',
#     custom_title="Channel Spend Analysis"
# )
# plt.show()

## Prior Specification

We illustrate pooled, unpooled, and partially pooled effects:
- `beta` (max contribution): hierarchical across geos with per-channel hyperpriors.
- `lam` (efficiency): fully pooled across geos (per channel only).
- `alpha` (adstock memory): unpooled across both geos and channels.

In [ ]:
# Beta prior: hierarchical over geos with per-channel mean and std
beta_prior = LogNormalPrior(
    mean=Prior("Gamma", mu=0.25, sigma=0.10, dims=("channel",)),
    std=Prior("Exponential", scale=0.10, dims=("channel",)),
    dims=("channel", "Groups"),
    centered=False,
)

# Saturation: partially pooled (beta hierarchical), lam fully pooled across geos
saturation = LogisticSaturation(
    priors={
        "beta": beta_prior,
        "lam": Prior("Gamma", mu=0.5, sigma=0.25, dims=("channel",)),
    }
)
print(saturation.model_config)


No_Saturation = NoSaturation(
        priors={"beta": Prior("HalfNormal", sigma=0.1, dims=("Groups", "channel"))}
    )

# Adstock: unpooled per (geo, channel)
adstock = GeometricAdstock(
    priors={"alpha": Prior("Beta", alpha=2, beta=5, dims=("Groups", "channel"))},
    l_max=8,
)
print(adstock.model_config)

# Other priors
model_config = {
    "intercept": Prior("Gamma", mu=0.5, sigma=0.25, dims="Groups"),
    "gamma_control": Prior("Normal", mu=0, sigma=0.5, dims="control"),
    "gamma_fourier": Prior("Normal", mu=0, sigma=0.2, dims=("Groups", "fourier_mode")),
    "likelihood": Prior(
        "TruncatedNormal",
        lower=0,
        sigma=Prior("HalfNormal", sigma=1.5),
        dims=("date", "Groups"),
    ),
}
model_config

## Model Definition

We define the multidimensional MMM, specify dims and scaling, and build to inspect the model graph.

In [ ]:
mmm = MMM(
    date_column=cols.date,
    target_column=cols.target,
    channel_columns=cols.channels,
    control_columns=cols.controls,
    dims=(cols.dimension,),
    scaling={
        "channel": {"method": "max", "dims": ()},
        "target": {"method": "max", "dims": ()},
    },
    adstock=adstock,
    saturation=No_Saturation,
    yearly_seasonality=2,
    model_config=model_config,
    apply_scaling =True,
    fit_intercept = True,
    log_transform_vars=["target", "channels"]
)

x_train = data_df.drop(columns=[cols.target]) 
y_train = data_df[cols.target]

# Build just to inspect; fit() would also build automatically
mmm.build_model(X=x_train, y=y_train)

# Add original-scale deterministic contributions
mmm.add_original_scale_contribution_variable(
    var=[
        "channel_contribution",
        "control_contribution",
        "intercept_contribution",
        "yearly_seasonality_contribution",
        
       
    ]
)

# Visualize the model graph (renders in notebooks)
try:
    pm.model_to_graphviz(mmm.model)
except Exception as e:
    print(f"Could not render model graph: {e}")

## Prior Predictive Checks

We draw from the prior and compare to the observed target distribution by geo.

In [ ]:
# Reload the visual_tools module to get the updated function
import importlib
import sys

# Remove from cache if present
if 'b2b_src.visual_tools' in sys.modules:del sys.modules['b2b_src.visual_tools']

# Re-import
from b2b_src.visual_tools import plot_multidimensional_prior_predictive_by_dim

print("Module reloaded successfully!")


In [ ]:
with mmm.model:
    prior = pm.sample_prior_predictive(random_seed=rng,  var_names=["y_original_scale"])



In [ ]:
# Use helper to plot prior predictive by dimension
from b2b_src.visual_tools import plot_multidimensional_prior_predictive_by_dim

# =============================================================================
# PRIOR PREDICTIVE CHECK
# =============================================================================

# OPTION 1: Automatic (shows "GEO_A", "GEO_B")
# fig = plot_multidimensional_prior_predictive_by_dim(prior, data_df)

# OPTION 2: Manual override - same label for all geos
fig = plot_multidimensional_prior_predictive_by_dim(
    prior, 
    data_df,
    dimension_label_override="NORTH REGION"
)



plt.show()

In [ ]:
print("Target transformation check:")
print(f"Original target (data_df['NetGMV']) mean: {data_df['NetGMV'].mean():.2f}")

# What was passed to build_model?
print(f"mmm.y mean: {mmm.y.mean():.2f}")

# What does the model see?
if 'target' in mmm.model.named_vars:
    print(f"target (pm.Data in model) mean: {mmm.model['target'].eval().mean():.2f}")

# What is target_data_scaled?
if hasattr(mmm, 'target_data_scaled'):
    print(f"target_data_scaled mean: {mmm.target_data_scaled.eval().mean():.2f}")


## Model Fitting

We fit the model and then draw posterior predictive samples for the training period.

In [ ]:
mmm.fit(
    X=x_train,
    y=y_train,
    chains=4,
    target_accept=0.95,
    random_seed=rng,
    nuts_sampler= "numpyro"
)

mmm.sample_posterior_predictive(
    X=x_train,
    random_seed=rng, var_names=["y_original_scale"]
)

# Quick diagnostics
print("Divergences per chain:")
try:
    display(mmm.idata.sample_stats.diverging.sum("draw"))
except Exception as e:
    print(e)

try:
    display(
        az.summary(
            mmm.idata,
            var_names=[
                "adstock_alpha",
                "gamma_control",
                "gamma_fourier",
                "intercept_contribution",
                "saturation_beta",
                "saturation_beta_mean",
                "saturation_beta_std",
                "saturation_lam",
                "y_sigma",
            ],
        )
    )
except Exception:
    pass

try:
    _ = az.plot_trace(
        data=mmm.idata,
        var_names=[
            "adstock_alpha",
            "gamma_control",
            "gamma_fourier",
            "intercept_contribution",
            "saturation_beta",
            "saturation_beta_mean",
            "saturation_beta_std",
            "saturation_lam",
            "y_sigma",
        ],
        compact=True,
        backend_kwargs={"figsize": (15, 15), "layout": "constrained"},
    )
    plt.gcf().suptitle("Model Trace", fontsize=16, fontweight="bold", y=1.03)
except Exception:
    pass

## Posterior Predictive Checks

We compare posterior predictive distributions to observed data for each geo.

In [ ]:
mmm.model.coords["Groups"]

In [ ]:
# =============================================================================
# POSTERIOR PREDICTIVE CHECK
# =============================================================================
from b2b_src.visual_tools import plot_posterior_predictive_by_geo

# OPTION 1: Automatic (shows "GEO_A", "GEO_B")
fig = plot_posterior_predictive_by_geo(mmm, data_df)

# OPTION 2: Manual override - same label for all geos
fig = plot_posterior_predictive_by_geo(
    mmm, 
    data_df,
    dimension_label_override="MARKET A"
)


plt.show()

# Get adjusted Component contributions 


In [ ]:
# =============================================================================
# CONFIGURATION: Use configuration from cols object
# =============================================================================

# Get channel and control columns from configuration
channel_cols = cols.channels
control_cols = cols.controls

# Get paid channels from configuration
# Assuming all channels in this dataset are paid channels
paid_channel_names = cols.channels

# Create spend mapping from configuration
# In this example, channel name matches spend column name
spend_mapping = {channel: channel for channel in cols.channels}

# Define baseline components for waterfall plots
baseline_components = ['intercept', 'seasonality']

# Create config dict for functions that need it
# Build channels list with type information
channels_config = [
    {'name': channel, 'spend_column': channel, 'type': 'paid'}
    for channel in cols.channels
]

config = {
    'data': {
        'columns': {
            'date_column': cols.date,
            'channel_columns': cols.channels,
            'control_columns': cols.controls,
            'channels': channels_config
        }
    }
}

In [ ]:
# =============================================================================
# STEP 3.1: Calculate Contributions (Multidimensional - ALL geos at once!)
# =============================================================================
print("Calculating contributions for all geos (vectorized)...")

contributions = get_individual_contributions_multidim(
    mmm=mmm,
    data=data_df,  # ← FIXED: Use data_df (defined in notebook)
    channel_cols=channel_cols,
    control_cols=control_cols,
    random_seed=seed
)

print(f"✓ Contributions calculated for {len(contributions['geos'])} geos")
print(f"  Geos: {contributions['geos']}")

# Verify shape
first_channel = list(contributions['channels'].values())[0]
print(f"  Shape: {first_channel.shape}")
print(f"  Dimensions: {first_channel.dims}")



In [ ]:
# =============================================================================
# ALL CONTRIBUTIONS PLOT
# =============================================================================

# OPTION 1: Automatic (shows "GEO_A", "GEO_B")
figs_by_geo = plot_all_contributions(
    contributions=contributions,
    mmm=mmm,
    data=data_df,
    use_log_scale=False
)

# # OPTION 2: Manual override - same label for all geos
# figs_by_geo = plot_all_contributions(
#     contributions=contributions,
#     mmm=mmm,
#     data=data_df,
#     use_log_scale=False,
#     dimension_label_override="GROUP"
# )

# OPTION 3: Custom titles + automatic dimension labels
# figs_by_geo = plot_all_contributions(
#     contributions=contributions,
#     mmm=mmm,
#     data=data_df,
#     use_log_scale=False,
#     custom_titles={
#         'combined': 'Q4 2024 Performance Analysis',
#         'individual': 'Detailed Component Breakdown'
#     }
# )

# OPTION 4: Custom titles + manual override
# figs_by_geo = plot_all_contributions(
#     contributions=contributions,
#     mmm=mmm,
#     data=data_df,
#     use_log_scale=False,
#     custom_titles={
#         'combined': 'Sales Attribution',
#         'individual': 'Component Details'
#     },
#     dimension_label_override="NORTH"
# )

print(f"✓ Plots generated for {len(figs_by_geo)} geos")

# Get ROAS

In [ ]:
# =============================================================================
# STEP 4: Calculate ROAS (Automatic per-geo calculation!)
# =============================================================================
print("Calculating ROAS for each geo...")

# Use pre-decorated version - automatically calculates for each geo
roas_by_geo = calculate_roas(
    contributions=contributions,
    data=data_df,
    channel_cols=paid_channel_names,
    target_column=cols.target,
    spend_mapping=spend_mapping
)

print(f"✓ ROAS calculated for {len(roas_by_geo)} geos")

# =============================================================================
# Display Results Per Geo
# =============================================================================
for geo in contributions['geos']:
    roas_results = roas_by_geo[geo]

    print("\n" + "=" * 80)
    print(f"ROAS RESULTS - {geo.upper()}")
    print("=" * 80)

    print("\nROAS by Channel:")
    print("-" * 40)
    for channel, roas in roas_results['roas'].items():
        print(f"  {channel}: {roas:.4f} (${roas:.2f} revenue per $1 spent)")

    print("\nTotal Spend by Channel:")
    print("-" * 40)
    for channel, spend in roas_results['spend'].items():
        print(f"  {channel}: ${spend:,.2f}")

    print("\nTotal Adjusted Contribution by Channel:")
    print("-" * 40)
    for channel, contrib in roas_results['adjusted_contributions_channels'].items():
        total_contrib = np.sum(contrib)
        print(f"  {channel}: ${total_contrib:,.2f}")

    # Verification
    print("\nVerification:")
    print("-" * 40)
    period_start = roas_results['period']['start']
    period_end = roas_results['period']['end']
    sum_all_adjusted = sum([np.sum(contrib) for contrib in roas_results['adjusted_contributions_all'].values()])

    # Get data for this geo
    data_geo = data_df[data_df[cols.dimension] == geo]
    sum_actual_y = data_geo.iloc[period_start:period_end+1][cols.target].sum()

    print(f"  Sum of all adjusted contributions: ${sum_all_adjusted:,.2f}")
    print(f"  Sum of actual Y: ${sum_actual_y:,.2f}")

In [ ]:
# =============================================================================
# STEP 5: Plot ROAS Analysis (Per Geo)
# =============================================================================
print("Generating ROAS analysis plots for each geo...")

# OPTION 1: Automatic dimension labels (RECOMMENDED)
# --------------------------------------------------
# Dimension labels will automatically show: "ROAS Analysis - GEO_A", "ROAS Analysis - GEO_B", etc.

# Note: We need to pass roas_by_geo (already calculated in cell above)
# The decorator will NOT work here because roas_by_geo is already split per geo

roas_analysis_figs_by_geo = {}

for geo in contributions['geos']:
    print(f"\n  Plotting ROAS analysis for {geo}...")
    
    # Extract per-geo data
    contrib_geo = slice_to_geo(contributions, geo)
    data_geo = data_df[data_df['Groups'] == geo].copy()
    roas_geo = roas_by_geo[geo]
    
    # Import the function
    from b2b_src.visual_tools import plot_roas_analysis as _plot_roas
    
    # # OPTION 1: With automatic dimension label
    # figs = _plot_roas(
    #     roas_results=roas_geo,
    #     contributions=contrib_geo,
    #     data=data_geo,
    #     confidence_intervals=(0.85, 0.90),
    #     baseline_components=baseline_components,
    #     dimension_label=geo  # Add this line for automatic labels (geo_a → GEO_A)
    # )
    
    # OPTION 2: With manual override
    figs = _plot_roas(
        roas_results=roas_geo,
        contributions=contrib_geo,
        data=data_geo,
        confidence_intervals=(0.85, 0.90),
        baseline_components=baseline_components,
        dimension_label='brand'  # Custom label for all plots
        ,
        custom_titles= "adasd"
    )
    
    # OPTION 3: No dimension label
    # figs = _plot_roas(
    #     roas_results=roas_geo,
    #     contributions=contrib_geo,
    #     data=data_geo,
    #     confidence_intervals=(0.85, 0.90),
    #     baseline_components=baseline_components
    # )
    
    roas_analysis_figs_by_geo[geo] = figs

print(f"\n✓ ROAS analysis plots generated for {len(roas_analysis_figs_by_geo)} geos")


In [ ]:
# =============================================================================
# STEP 6: Calculate ROAS Time Series (Automatic per-geo!)
# =============================================================================
print("Calculating ROAS time series for each geo...")

# Use pre-decorated version - automatically calculates for each geo
roas_ts_by_geo = calculate_roas_time_series(
    contributions=contributions,
    data=data_df,
    channel_cols=paid_channel_names,
    target_column=cols.target,
    spend_mapping=spend_mapping
)

print(f"✓ ROAS time series calculated for {len(roas_ts_by_geo)} geos")

# =============================================================================
# Display Results Per Geo
# =============================================================================
for geo in contributions['geos']:
    roas_ts_results = roas_ts_by_geo[geo]

    print("\n" + "="*80)
    print(f"ROAS TIME SERIES RESULTS - {geo.upper()}")
    print("="*80)
    print(f"\nChannels processed: {roas_ts_results['channel_cols']}")
    print(f"Number of dates: {len(roas_ts_results['dates'])}")
    print(f"Date range: {roas_ts_results['dates'][0]} to {roas_ts_results['dates'][-1]}")

    for channel in roas_ts_results['channel_cols']:
        roas_samples = roas_ts_results['roas_time_series'][channel]
        print(f"\n{channel}:")
        print(f"  ROAS array shape: {roas_samples.shape} (n_samples, n_dates)")
        print(f"  Mean ROAS: {np.nanmean(roas_samples):.4f}")

# =============================================================================
# Plot ROAS Time Series (Per Geo)
# =============================================================================
print("\n" + "="*80)
print("GENERATING ROAS TIME SERIES PLOTS...")
print("="*80)

roas_ts_figs_by_geo = {}

for geo in contributions['geos']:
    print(f"\n  Plotting ROAS time series for {geo}...")

    # Import the function
    from b2b_src.visual_tools import plot_roas_time_series as _plot_roas_ts

    # # OPTION 1: With automatic dimension label (geo_a → "GEO_A")
    # plot_results = _plot_roas_ts(
    #     roas_results=roas_ts_by_geo[geo],
    #     confidence_levels=(0.95, 0.99),
    #     show_mean_line=True,
    #     figsize=(14, 6),
    #     dimension_label=geo  # Add this line for automatic labels
    # )
    
    # OPTION 2: With manual override (same label for all geos)
    plot_results = _plot_roas_ts(
        roas_results=roas_ts_by_geo[geo],
        confidence_levels=(0.95, 0.99),
        show_mean_line=True,
        figsize=(14, 6),
        dimension_label='North Region'  # Custom label
    )
    
    # OPTION 3: No dimension label
    # plot_results = _plot_roas_ts(
    #     roas_results=roas_ts_by_geo[geo],
    #     confidence_levels=(0.95, 0.99),
    #     show_mean_line=True,
    #     figsize=(14, 6)
    # )

    roas_ts_figs_by_geo[geo] = plot_results

plt.show()

print(f"\n✓ ROAS time series plots generated for {len(roas_ts_figs_by_geo)} geos")

In [ ]:
# =============================================================================
# STEP 7: Calculate Marginal ROAS (Manual per-geo)
# =============================================================================
# Note: calculate_marginal_roas doesn't use contributions, so we call it manually per-geo

print("Calculating marginal ROAS for each geo...")

mroas_by_geo = {}

for geo in contributions['geos']:
    print(f"\n  Calculating marginal ROAS for {geo}...")

    # Filter data for this geo and RESET INDEX (important!)
    data_geo = data_df[data_df[cols.dimension] == geo].copy().reset_index(drop=True)

    # Use original function directly
    from b2b_src.contributions import calculate_marginal_roas as _calc_mroas

    mroas_results = _calc_mroas(
        mmm=mmm,
        data=data_geo,
        config=config,
        marginal_increase=0.01,  # 1% increase
        random_seed=seed
    )

    mroas_by_geo[geo] = mroas_results

print(f"\n✓ Marginal ROAS calculated for {len(mroas_by_geo)} geos")

# =============================================================================
# Display Results Per Geo
# =============================================================================
for geo in contributions['geos']:
    mroas_results = mroas_by_geo[geo]

    print("\n" + "="*80)
    print(f"MARGINAL ROAS RESULTS - {geo.upper()}")
    print("="*80)

    print("\nMarginal ROAS by Channel:")
    print("-" * 40)
    for channel, mroas in mroas_results['mroas'].items():
        print(f"  {channel}: {mroas:.4f}")

    print("\nIncremental Sales by Channel:")
    print("-" * 40)
    for channel, inc_sales in mroas_results['incremental_sales'].items():
        print(f"  {channel}: ${inc_sales:,.2f}")

In [ ]:
# # =============================================================================
# # STEP 8: Analyze Revenue vs Spend Levels (Manual per-geo)
# # =============================================================================
# # Note: This function is not yet decorated in multi_src, so we process manually

# print("Analyzing revenue vs spend levels for each geo...")

# results_by_geo = {}

# for geo in contributions['geos']:
#     print(f"\n  Analyzing {geo}...")

#     # Filter data for this geo and RESET INDEX (important!)
#     data_geo = data_df[data_df['Groups'] == geo].copy().reset_index(drop=True)

#     # Analyze for this geo
#     results_geo = analyze_revenue_vs_spend_levels(
#         mmm=mmm,
#         data=data_geo,
#         config=config,
#         random_seed=327
#     )

#     results_by_geo[geo] = results_geo

# print(f"\n✓ Analysis completed for {len(results_by_geo)} geos")

In [ ]:
# =============================================================================
# Plot Response Curves (Per Geo)
# =============================================================================
# print("Generating response curves for each geo...")

# for geo in contributions['geos']:
#     print(f"\n  Plotting response curves for {geo}...")

#     # OPTION 1: With automatic dimension label (geo_a → "GEO_A")
#     fig, ax = plot_response_curves(results_by_geo[geo], dimension_label=geo)
#     plt.show()
    
    # OPTION 2: With manual override (custom label)
    # fig, ax = plot_response_curves(
    #     results_by_geo[geo],
    #     dimension_label="North Region"
    # )
    # plt.show()
    
    # OPTION 3: Custom title + automatic dimension label
    # fig, ax = plot_response_curves(
    #     results_by_geo[geo],
    #     custom_title="Revenue Analysis",
    #     dimension_label=geo
    # )
    # plt.show()
    
    # OPTION 4: Custom title + manual override
    # fig, ax = plot_response_curves(
    #     results_by_geo[geo],
    #     custom_title="Spend Optimization Curves",
    #     dimension_label="Market A"
    # )
    # plt.show()

# print(f"\n✓ Response curves generated for {len(results_by_geo)} geos")
